In [57]:
import pandas as pd

In [74]:
import os

os.makedirs('tracking-progress-elementary', exist_ok=True)
os.chdir('tracking-progress-elementary')


In [75]:
import sys
from pathlib import Path
import subprocess

venv_dir = Path(".venv")
if not venv_dir.exists():
    subprocess.run([sys.executable, "-m", "venv", str(venv_dir)], check=True)

venv_python = venv_dir / ("Scripts" if sys.platform == "win32" else "bin") / "python"
subprocess.run(
    [str(venv_python), "-m", "pip", "install", "pandas", "numpy", "matplotlib", "seaborn", "jupyter", "ipykernel"],
    check=True,
)

with open("requirements.txt", "w", encoding="utf-8") as f:
    subprocess.run([str(venv_python), "-m", "pip", "freeze"], check=True, stdout=f)

In [104]:
import sqlite3, pandas as pd
from pathlib import Path
import importlib.util
import sys

# Locate a local clean.py in the notebook/project tree (avoid site-packages)
clean_path = None
for p in Path('.').rglob('clean.py'):
    if 'site-packages' in str(p):
        continue
    clean_path = p
    break

if clean_path is None:
    # fallback to standard import (will raise the original error if wrong module is on sys.path)
    for parent_dir in [Path.cwd()] + list(Path.cwd().parents):
        candidate = parent_dir / "clean.py"
        if candidate.exists() and "site-packages" not in str(candidate):
            spec = importlib.util.spec_from_file_location("clean", str(candidate))
            clean = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(clean)
            clean_diagnostics = clean.clean_diagnostics
            clean_school_summary = clean.clean_school_summary
            clean_school_improvement = clean.clean_school_improvement
            break
else:
    spec = importlib.util.spec_from_file_location("clean", str(clean_path))
    clean = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(clean)
    clean_diagnostics = clean.clean_diagnostics
    clean_school_summary = clean.clean_school_summary
    clean_school_improvement = clean.clean_school_improvement

In [12]:
import sqlite3
from pathlib import Path

db_path = "schooldata.db"  # adjust this path to your actual sqlite file
conn = sqlite3.connect(db_path)

query = "SELECT COUNT(*) FROM diagnostic-results_dummy-database_with-behavior_1000rows;"
db_path = "schooldata.db"  # adjust this path to your actual sqlite file
conn = sqlite3.connect(db_path)

query = 'SELECT COUNT(*) FROM "diagnostic-results_dummy-database_with-behavior_1000rows";'
db_path = "schooldata.db"
conn = sqlite3.connect(db_path)

tables = conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;"
).fetchall()
print("Available tables:", [t[0] for t in tables])

table_name = "diagnostic-results_dummy-database_with-behavior_1000rows"
if (table_name,) not in tables:
    db_path = Path("schooldata.db")
    if not db_path.exists():
        raise FileNotFoundError(f"Database file not found: {db_path.resolve()}")

    conn = sqlite3.connect(db_path)

    tables = conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;"
    ).fetchall()
    table_names = [t[0] for t in tables]
    print("Available tables:", table_names)

    table_name = "diagnostic-results_dummy-database_with-behavior_1000rows"
    if table_name not in table_names:
        db_candidates = [
            db_path,
            Path.cwd() / "schooldata.db",
            Path.cwd().parent / "schooldata.db",
            Path.cwd() / "Notebooks" / "schooldata.db",
            Path.cwd().parent / "Notebooks" / "schooldata.db",
        ]
        db_path = next((p for p in db_candidates if p.exists()), None)

        if db_path is None:
            raise FileNotFoundError(
            f"Database file not found. Checked: {[str(p.resolve()) for p in db_candidates]}"
            )

        conn = sqlite3.connect(str(db_path))
        tables = conn.execute(
            "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;"
        ).fetchall()
        table_names = [t[0] for t in tables]
        print("Available tables:", table_names)

        if table_name not in table_names:
            # Try the known database candidates first. If none contains the target table,
            # search the project tree for any SQLite database that does.
            found_db = None
            found_table_names = []

            for candidate in db_candidates:
                candidate = Path(candidate)
                if not candidate.exists():
                    continue
                try:
                    with sqlite3.connect(str(candidate)) as test_conn:
                        test_tables = test_conn.execute(
                            "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;"
                        ).fetchall()
                        test_table_names = [t[0] for t in test_tables]
                        if table_name in test_table_names:
                            found_db = candidate
                            found_table_names = test_table_names
                            break
                except Exception as exc:
                    print(f"Could not inspect {candidate}: {exc}")

            if found_db is None:
                project_root = Path.cwd().resolve()
                for candidate in sorted(project_root.rglob("*.db")):
                    try:
                        with sqlite3.connect(str(candidate)) as test_conn:
                            test_tables = test_conn.execute(
                                "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;"
                            ).fetchall()
                            test_table_names = [t[0] for t in test_tables]
                            if table_name in test_table_names:
                                found_db = candidate
                                found_table_names = test_table_names
                                break
                    except Exception:
                        continue

Available tables: []
Available tables: []
Available tables: []


In [14]:
import sqlite3
import pandas as pd
# SciPy is not required in this cell, so remove the unused import to avoid ModuleNotFoundError

DB_PATH = "db/capstone.db"


def get_connection():
    """Open a connection to the capstone SQLite database."""
    return sqlite3.connect(DB_PATH)


def summary_stats(df: pd.DataFrame, group_col: str, value_col: str) -> pd.DataFrame:
    """
    Return count/mean/median/std of value_col, grouped by group_col.
    Reused for both the student thread (e.g. group by at_risk_flag)
    and the school thread (e.g. group by federal_classification).
    """
    result = (
        df.groupby(group_col)[value_col]
        .agg(count="count", mean="mean", median="median", std="std")
        .reset_index()
        .round(2)
    )
    return result

In [15]:
def load_student_diagnostics() -> pd.DataFrame:
    conn = get_connection()
    df = pd.read_sql_query("SELECT * FROM student_diagnostics", conn)
    conn.close()
    return df


def load_school_accountability() -> pd.DataFrame:
    conn = get_connection()
    df = pd.read_sql_query(
        """
        SELECT s.*, i.federal_classification, i.years_identified
        FROM school_accountability s
        LEFT JOIN school_improvement i
          ON s.school_code = i.school_code
         AND s.school_year = i.school_year
        """,
        conn,
    )
    conn.close()
    return df

In [16]:
def compute_correlations(student_df: pd.DataFrame, school_df: pd.DataFrame) -> dict:
    """
    Correlation between behavior_index and growth_gap (student thread),
    and between climate_safety_rate and reading_math_rate (school thread).
    """
    student_clean = student_df.dropna(subset=["behavior_index", "growth_gap"])
    r_student, p_student = stats.pearsonr(
        student_clean["behavior_index"], student_clean["growth_gap"]
    )

    school_clean = school_df.dropna(subset=["climate_safety_rate", "reading_math_rate"])
    r_school, p_school = stats.pearsonr(
        school_clean["climate_safety_rate"], school_clean["reading_math_rate"]
    )

    return {
        "student_behavior_vs_growth": {"r": round(r_student, 3), "p": round(p_student, 4)},
        "school_climate_vs_reading_math": {"r": round(r_school, 3), "p": round(p_school, 4)},
    }

In [25]:
from pathlib import Path
import sqlite3

if __name__ == "__main__":
    DB_PATH = Path("db") / "capstone.db"
    if not DB_PATH.exists():
        for base in [Path.cwd(), *Path.cwd().parents]:
            candidate = base / "db" / "capstone.db"
            if candidate.exists():
                DB_PATH = candidate
                break
        else:
            if __name__ == "__main__":
                DB_PATH = Path("db") / "capstone.db"
                if not DB_PATH.exists():
                    for base in [Path.cwd(), *Path.cwd().parents]:
                        candidate = base / "db" / "capstone.db"
                        if candidate.exists():
                            DB_PATH = candidate
                            break
                    else:
                        for base in [Path.cwd(), *Path.cwd().parents]:
                            candidate = base / "schooldata.db"
                            if candidate.exists():
                                DB_PATH = candidate
                                break
                        else:
                            raise FileNotFoundError(
                                "Could not find database file. Checked for db/capstone.db and schooldata.db in cwd and parent directories."
                            )

    def get_connection():
        return sqlite3.connect(str(DB_PATH))
    db_candidates = [
        Path.cwd() / "schooldata.db",
        Path.cwd().parent / "schooldata.db",
        Path.cwd() / "Notebooks" / "schooldata.db",
        Path.cwd().parent / "Notebooks" / "schooldata.db",
    ]
    DB_PATH = next((p for p in db_candidates if p.exists()), None)
    if DB_PATH is None:
        raise FileNotFoundError(
            "Could not find schooldata.db. Checked: "
            f"{[str(p.resolve()) for p in db_candidates]}"
        )

In [29]:
import os
import subprocess
import sys
from pathlib import Path

for base in [Path.cwd(), *Path.cwd().parents]:
    script_path = base / "src" / "analysis.py"
    if script_path.exists():
        if "DB_PATH" in globals() and DB_PATH is not None:
            os.environ["DB_PATH"] = str(DB_PATH)
        subprocess.run([sys.executable, str(script_path)], cwd=base, check=True)
        break
else:
    search_roots = []

    if "DB_PATH" in globals() and DB_PATH is not None:
        db_path = Path(DB_PATH).resolve()
        if db_path.exists():
            search_roots.extend([db_path.parent, db_path.parent.parent])

    cwd = Path.cwd().resolve()
    search_roots.extend([cwd, *cwd.parents])

    seen = set()
    for base in search_roots:
        base = base.resolve()
        if base in seen:
            continue
        seen.add(base)

        script_path = base / "src" / "analysis.py"
        if script_path.exists():
            if "DB_PATH" in globals() and DB_PATH is not None:
                os.environ["DB_PATH"] = str(DB_PATH)
            subprocess.run([sys.executable, str(script_path)], cwd=base, check=True)
            break